# CMLP GMLP SAOCOM EMI phase estimation

In [ ]:
"""
File Name: cmlp_SAOCOM_EMI.ipynb

Author:
    Francesco De Zan
    delta phi remote sensing GmbH

Date: March 2026

Description:
------------
This script performs the generation of CMLP for a stack of CSLCs from the SAOCOM sensor. 

Usage:
------
To use this script, define the following parameters:
    
    - emi_file_name
    
"""

In [ ]:
import numpy as np
import datatree
import xarray as xr
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil

In [2]:
import rosel.io
import rosel.computation
import rosel.shiroma

In [18]:
type, n_jobs, job_memory = ["LOCAL", 16, None]
client = rosel.computation.create_client(type, n_jobs, job_memory)

2025-04-17 15:46:04 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=3997717222>


In [ ]:
base_dir = "/media/data/SAR/L2/Naples/DSC/S4/"
channel_dir = "SAOCOM_DSC_S4_20250220_POLARIZATION_VV/"
emi_file_name = base_dir +"cmlcs_emi.zarr"

In [ ]:
# selecting window sizes (rg, az)
w_rg = 10
w_az = 20
multilook = {"azimuth_time": w_az, "slant_range_time": w_rg}

# subsampling for geocoding: default is sampled @ 10m x 10m, we want 50m x 50m
subsampling = 5

In [23]:
# opening the CSLC dataset
#ds = datatree.open_datatree(base_dir + 'slcs.zarr', engine='zarr', chunks={})
ds = datatree.open_datatree(cslc_file_name, engine='zarr', chunks={})

cstack = ds.coreg.cstack
N = len(cstack)

In [ ]:
# initialization for full-scene version
n_lines, n_samples = cstack[0].shape
n_rg = np.floor_divide(n_samples, w_rg)
n_az = np.floor_divide(n_lines, w_az)

In [ ]:
def emi_core(C_matrix) :
    try:
        W = np.linalg.inv(np.abs(C_matrix))
        eigenvalues, eigenvectors = np.linalg.eigh(C_matrix * W)
        v = eigenvectors[:,0]
    except:
        v = np.zeros([N])
    v = v / np.abs(v)
    v = v * np.conj(v[0])
    return np.conj(v)

In [19]:
def emi_phase_estimation(window):
    data = np.zeros([N, w_rg*w_az], dtype=np.complex64)
    ind = np.full(N,True)
    interf_emi = np.full(N, np.NaN, dtype=np.complex64) # default output

    #loading data and checking for non-valid acquisitions
    for n in range(0,N):
        data[n,:] = window[n,:,:].flatten()
        #ind[n] = (~np.isnan(data[n,:])).all() # check that all data are valid
        ind[n] = (data[n,:] != 0).all() # check that all data are valid
    
    N_ = sum(ind) # number of valid acquisitions
    if N_ != 0:
        data = data[ind,:] # eliminate non-valid acquisitions
        cov = np.matmul(data.conj(), data.T)
        interf_emi_ = emi_core(cov + 0.01*np.linalg.eigvalsh(cov)[N_-1]*np.eye(N_))

        # filling expanded output
        interf_emi[ind] = interf_emi_
        
    return interf_emi

In [20]:
# EMI phase estimation
temp = cstack.coarsen(azimuth_time=w_az, slant_range_time=w_rg, boundary='trim').construct(
    azimuth_time=("az_coarse", "az_window"), slant_range_time=("rg_coarse", "rg_window"), keep_attrs=True)
temp = temp.chunk(dict(acquisition_date=N)) # no chunks allows in core dimensions!

interf_emi = xr.apply_ufunc(
    emi_phase_estimation, 
    temp,
    input_core_dims=[["acquisition_date", "az_window", "rg_window"]],
    output_core_dims=[["acquisition_date"]],
    output_dtypes=[temp.dtype],
    vectorize=True,
    dask="parallelized"
)

In [21]:
interf_emi = interf_emi.compute()

2025-03-26 13:02:36,992 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 3.00 GiB -- Worker memory limit: 3.72 GiB
2025-03-26 13:02:39,789 - distributed.worker.memory - WARNING - Worker is at 75% memory usage. Resuming worker. Process memory: 2.80 GiB -- Worker memory limit: 3.72 GiB
2025-03-26 13:03:13,669 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 2.99 GiB -- Worker memory limit: 3.72 GiB
2025-03-26 13:03:14,113 - distributed.worker.memory - WARNING - Worker is at 75% memory usage. Resuming worker. Process memory: 2.81 GiB -- Worker memory limit: 3.72 GiB
2025-03-26 13:04:25,574 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 2.98 GiB -- Worker memory limit: 3.72 GiB
2025-03-26 13:04:26,279 - distributed.worker.memory - WARNING - Worker is at 74% memory usage. Resuming worker. Process memory: 2.78 GiB -- Worker mem

In [22]:
interf_emi = interf_emi.transpose('acquisition_date', 'az_coarse', 'rg_coarse')

In [ ]:
# collecting and formatting output
emi_ds = xr.Dataset(
    data_vars=dict(
        phase=(["acquisition_date", "azimuth_time", "slant_range_time"], interf_emi.data),
    ),

    coords=dict(
        azimuth_time=("azimuth_time", cstack[0].azimuth_time.coarsen(azimuth_time=w_az, boundary='trim').mean().data),
        slant_range_time=("slant_range_time", cstack[0].slant_range_time.coarsen(slant_range_time=w_rg, boundary='trim').mean().data),
        acquisition_date=("acquisition_date", cstack.acquisition_date.data),
    ),

    attrs=dict(description="Co-registered Multi-look Complex (EMI)"),
)

In [25]:
emi_file_name

'/media/data/SAR/L2/Naples/DSC/S3/cmlcs_emi.zarr'

In [26]:
#saving
emi_ds.to_zarr(emi_file_name)

In [ ]:
# loading EMI CMLC
##ds_emi = datatree.open_datatree(emi_file_name, engine='zarr', chunks={})